<div style="text-align:left;">
    <span style="
        display:inline-block;
        background-color:#4169E1;
        color:white;
        padding:10px 20px;
        border-radius:8px;
        font-size:45px;
        font-weight:bold;
    ">
        Deployment Preparation
    </span>
</div>

**Author:** Sarra Chouk 

**Student ID:** 60300372

**Project:** EHR Mortality Risk Prediction  

**Dataset:** MIMIC-IV

**Date:** April 4, 2026  

---

### **Objective**
To packages the final model for batch deployment.

### **Setup and Library Imports**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to sys.path:")
print(PROJECT_ROOT)

Project root added to sys.path:
/Users/sarrachouk/Desktop/ehr-mortality-prediction


In [10]:
import importlib
import src.models.deployment_utils as deployment_utils
importlib.reload(deployment_utils)

from src.models.deployment_utils import (
    load_model_ready_train_data,
    fit_calibrated_xgb,
    save_packaged_model,
)

### **Define Paths**

In [7]:
TRAIN_FULL_PATH = "../data/processed/splits/train_full.csv"
PACKAGED_MODEL_DIR = Path("../deployment/packaged_model")
PACKAGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Train full path:", Path(TRAIN_FULL_PATH).resolve())
print("Packaged model dir:", PACKAGED_MODEL_DIR.resolve())

Train full path: /Users/sarrachouk/Desktop/ehr-mortality-prediction/data/processed/splits/train_full.csv
Packaged model dir: /Users/sarrachouk/Desktop/ehr-mortality-prediction/deployment/packaged_model


### **Load the Model-ready Training Data**

In [8]:
X_train, y_train = load_model_ready_train_data(TRAIN_FULL_PATH)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
display(X_train.head())

X_train shape: (380461, 49)
y_train shape: (380461,)


,age,gender,ed_los_hours,came_from_ed,admission_hour,admission_day_of_week,num_prev_visits,time_since_last_visit_days,avg_time_between_visits_days,avg_prev_los,...,admission_type_urgent,admission_type_other,marital_status_single,marital_status_divorced,marital_status_widowed,marital_status_unknown,race_black,race_asian,race_hispanic_latino,race_other
0,65,1,6.33,1,14,5,0,0.00,0.0,0.00,...,0,1,1,0,0,0,0,0,0,0
1,65,1,16.62,1,2,5,1,27.52,0.0,26.21,...,0,1,1,0,0,0,0,0,0,0
2,30,0,0.00,0,22,1,0,0.00,0.0,0.00,...,1,0,1,0,0,0,0,0,1,0
3,30,0,15.30,1,3,4,1,9.23,0.0,5.67,...,1,0,1,0,0,0,0,0,1,0
4,22,0,28.75,1,21,6,0,0.00,0.0,0.00,...,0,1,1,0,0,0,0,0,1,0


### **Fit the Final Tuned XGBoost Model**

In [11]:
final_model = fit_calibrated_xgb(X_train, y_train)
print("Final calibrated model fitted successfully.")

Final calibrated model fitted successfully.


### **Save Packaged Model**

In [14]:
save_packaged_model(
    final_model,
    PACKAGED_MODEL_DIR,
    input_feature_columns=X_train.columns.tolist(),
)

print("Packaged model saved successfully.\n")

for p in PACKAGED_MODEL_DIR.iterdir():
    print(p.name)

Packaged model saved successfully.

input
output
deployment_metadata.json
xgb_end_to_end_batch_model.joblib
expected_model_input_columns.json


### **Sanity Check**

In [15]:
with open(PACKAGED_MODEL_DIR / "deployment_metadata.json", "r", encoding="utf-8") as f:
    deployment_metadata = json.load(f)

deployment_metadata

{'model_file': 'xgb_end_to_end_batch_model.joblib',
 'threshold': 0.07,
 'best_params': {'colsample_bytree': 0.8,
  'gamma': 0,
  'learning_rate': 0.05,
  'max_depth': 8,
  'min_child_weight': 5,
  'n_estimators': 200,
  'subsample': 1.0},
 'calibration_method': 'sigmoid',
 'expected_model_input_columns_file': 'expected_model_input_columns.json',
 'prediction_probability_column': 'predicted_probability',
 'prediction_label_column': 'predicted_label'}